In [ ]:
%pip install tabpfn
%pip install tabpfn-client
%pip install matplotlib
%matplotlib inline
%pip install rich

import pandas as pd
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier
from sklearn.metrics import accuracy_score

# Datensatz laden und bereinigen


In [ ]:
# 1. Datensatz aus Github-URL als CSV einlesen
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
data = pd.read_csv(url)

# 3. Spaltennamen ins Deutsche übersetzen
spalten_mapping = {
    'gender': 'Geschlecht',
    'SeniorCitizen': 'Senior',
    'Partner': 'Partner',
    'Dependents': 'Kinder_im_Haushalt',
    'tenure': 'Vertragsdauer_Monate',
    'PhoneService': 'Telefon_Dienst',
    'MultipleLines': 'Mehrfachleitungen',
    'InternetService': 'Internetdienst',
    'OnlineSecurity': 'Online_Sicherheit',
    'OnlineBackup': 'Online_Backup',
    'DeviceProtection': 'Geräteschutz',
    'TechSupport': 'Technischer_Support',
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filme',
    'Contract': 'Vertragsart',
    'PaperlessBilling': 'Papierlose_Rechnung',
    'PaymentMethod': 'Zahlungsmethode',
    'MonthlyCharges': 'Monatliche_Kosten_Euro',
    'TotalCharges': 'Gesamtkosten_Euro',
    'Churn': 'Kuendigung'
}
data = data.rename(columns=spalten_mapping)


data.head()


# 2.1 TotalCharges von Text in echte Zahlen umwandeln
data['Gesamtkosten_Euro'] = pd.to_numeric(data['Gesamtkosten_Euro'], errors='coerce').fillna(0)

# 2.2 customerID löschen (das ist nur ein Name, kein Merkmal fürs Modell)
data = data.drop(columns=['customerID'])

# 2.3 Zielvariable Churn (Yes/No) in 1 und 0 umwandeln
data['Kuendigung'] = data['Kuendigung'].map({'Yes': 1, 'No': 0})


# 3. Features und Zielvariable trennen
X = data.drop(columns=['Kuendigung'])
y = data['Kuendigung']


display(data)

### 🛠️ Backend-Auswahl: Lokale Ausführung vs. Cloud-API

TabPFN bietet zwei Betriebsmodi für flexible Einsatzszenarien:

1. **Lokaler Modus (Standard & Datenschutz-Fokus):**
   * **Vorteil**: Die Berechnungen und das Training finden vollständig auf deiner Hardware (oder der Colab-VM) statt. Deine Tabellendaten werden niemals an externe Server übertragen (vollständig datenschutzkonform für sensible Daten).
   * **Hinweis**: Erfordert einmalig einen kostenlosen Prior Labs Account und die Lizenzbestätigung beim ersten Download der Modellgewichte.
2. **Cloud-API Modus (`tabpfn_client`):**
   * **Vorteil:** Verlagert die Berechnung auf die Server von Prior Labs.
   * **Ideal für:** Schwächere Laptops oder schnelle Testumgebungen ohne dedizierte Hardware.

> **Hinweis zur Einrichtung:**
> Für beide Nutzungsverfahren ist ein kostenloser Account bei Prior Labs erforderlich. Führe das folgende Skript aus und befolge die Anweisungen

HIER: **Betriebsart auswählen**

In [ ]:
# True  = Über die Cloud-API (schont schwache Laptops, erfordert einen Prior Labs Token)
# False = Lokal auf dem Computer (vollständig datenschutzkonform, läuft komplett offline)
USE_CLOUD_API = True  

In [ ]:
import os
import getpass
from rich.console import Console
from rich.panel import Panel
from rich.rule import  Rule

console = Console()

# --- Automatische Authentifizierung (für lokal & client gleichermaßen nötig) ---
tabpfn_token = None
try:
    from google.colab import userdata
    tabpfn_token = userdata.get("TABPFN_TOKEN")
    if tabpfn_token:
      os.environ["TABPFN_TOKEN"] = tabpfn_token
except Exception:
  pass

if not tabpfn_token and not os.environ.get("TABPFN_TOKEN"):
  console.print(
      Panel(
          "So erhältst du deinen kostenlosen API-Key:\n\n"
          "  1. Gehe auf [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] und registriere dich\n"
          "  2. Akzeptiere die Lizenz unter [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
          "  3. Kopiere deinen API-Key von [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
          "[bold yellow]Tipp:[/bold yellow] Trage den Key einmalig links in Colab unter dem "
          "[bold cyan]Schlüsselsymbol (Secrets)[/bold cyan] als [bold cyan]TABPFN_TOKEN[/bold cyan] ein.",
          title="[bold]🔑 Prior Labs API-Key erforderlich[/bold]",
          border_style="blue",
      )
  )
  while not tabpfn_token:
    token_input = getpass.getpass("Füge deinen TABPFN_TOKEN ein und drücke Enter: ")
    if token_input.strip():
      tabpfn_token = token_input.strip()
      os.environ["TABPFN_TOKEN"] = tabpfn_token
    else:
      console.print("[red]Der Token darf nicht leer sein. Bitte versuche es erneut.[/red]")

# --- Backend-Auswahl ---
if USE_CLOUD_API:
  try:
    from tabpfn_client import TabPFNClassifier, init, set_access_token

    set_access_token(os.environ.get("TABPFN_TOKEN"))
    init()
    classifier = TabPFNClassifier()
    console.print(
        "✅ [bold green]Modus: Cloud-API aktiv (Berechnung auf Prior Labs"
        " Servern).[/bold green]"
    )
  except ImportError:
    console.print(
        "❌ [bold red]Fehler: 'tabpfn_client' ist nicht installiert. Bitte"
        " führe '!pip install tabpfn-client' aus.[/bold red]"
    )
else:
  # Lokale Variante (Datenschutz-Fokus für KMUs)
  from tabpfn import TabPFNClassifier

  classifier = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
  console.print("✅ [bold green]Modus: Lokale Ausführung aktiv[/bold green]")

# Modelltraining mit TabPFN

In [ ]:
# 1. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TabPFN initialisieren (läuft lokal auf der CPU)
classifier = TabPFNClassifier(ignore_pretraining_limits=True)

print("Modell wird trainiert...")
classifier.fit(X_train, y_train)

#clear_output(wait=True)

# 3. Vorhersage und Genauigkeit berechnen
y_pred_tabpfn = classifier.predict(X_test)
rf_risiko = classifier.predict_proba(X_test_bereit)[:, 1]

def bewerte_modell(name, y_test, y_pred, y_risiko):
    return {
        "Modell": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_risiko),
    }

tabpfn_results = bewerte_modell("TabPFN", y_test, y_pred_tabpfn, rf_risiko)


# BENCHMARK: Vergleich mit Random Forest
**Warum Random Forest?** 
    Er gilt im Data Science als einer der robustesten und beliebtesten Standard-Klassifikatoren für tabellarische Daten. Er liefert solide Ergebnisse, ohne dass man stundenlang an Parametern schrauben muss – genau wie TabPFN. Das macht ihn zum idealen Vergleichspartner.


- **Die technische Hürde (Datenaufbereitung):** 
    Ein klassischer Random Forest kann rein mathematisch **nur mit Zahlen** rechnen. Textspalten wie `"Male"` / `"Female"` oder Verträge können von dem Algorithmus nicht direkt verarbeitet werden.
- **Die Lösung (Pipeline & ColumnTransformer):** 
    Mithilfe einer `Pipeline` und eines `ColumnTransformer` automatisieren wir diesen Schritt: Textspalten werden automatisch erkannt und per `OrdinalEncoder` in Zahlen übersetzt, während reine Zahlen-Spalten einfach unverändert durchgelassen werden. Das hält den Workflow modular und schützt vor Datenlecks.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1. Alle Spalten mit kategorischen Daten identifizieren
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Den ColumnTransformer einrichten
column_transformer = ColumnTransformer(
    transformers=[
        ('CatToNum', OrdinalEncoder(), categorical_cols)
    ],
    remainder='passthrough'  # Numerische Spalten einfach unverändert durchlassen
)

# 3. Pipeline erstellen: Erst umwandeln, dann Random Forest trainieren
rf_pipeline = make_pipeline(column_transformer, RandomForestClassifier(random_state=42))

print("Random Forest Pipeline wird trainiert...")
rf_pipeline.fit(X_train, y_train)

# 4. Vorhersage ausführen
y_pred_rf = rf_pipeline.predict(X_test)
rf_risiko = rf_pipeline.predict_proba(X_test)[:, 1]

# clear_output(wait=True) 


rf_results = bewerte_modell("Random Forest", y_test, y_pred_rf, rf_risiko)

# direkter Vergleich

In [ ]:
# Direkter Leistungsvergleich
print("--- MODELLVERGLEICH ---")
print(f"Random Forest Genauigkeit: {acc_rf:.4f}")
print(f"TabPFN Genauigkeit: {acc_tabpfn:.4f}")

# 5. Visueller Vergleich als Balkendiagramm
results_df = pd.DataFrame({
    'Model': ['Random Forest', 'TabPFN'],
    'Accuracy': [acc_rf, acc_tabpfn]
})

ax = results_df.plot(x='Model', y='Accuracy', kind='bar', ylim=(0.7, 0.9), figsize=(8, 4), legend=False, color=['skyblue', 'salmon'])
ax.set_title("Modellvergleich: Einfacher Train-Test-Split")
ax.set_ylabel("Genauigkeit (Accuracy)")

- **Ergebnisse:** 
    Während der Random Forest ein stabiles, bewährtes Arbeitstier ist, zeigt der direkte Vergleich, wie sich moderne Ansätze (wie TabPFN) schlagen. Oft liefert TabPFN direkt out-of-the-box extrem starke Ergebnisse, ohne dass man sich um komplexe Hyperparameter-Suche (Grid Search) kümmern muss.
    
- **Fazit für die Praxis:** 
    Für KMUs ist oft nicht nur die reine Genauigkeit entscheidend, sondern auch die **Zeit** und der **Aufwand**, die man in die Datenaufbereitung und das Tuning stecken muss. Pipelines helfen uns dabei, diesen Prozess reproduzierbar und fehlerfrei zu halten.

## Welche Kunden sollte das Unternehmen zuerst kontaktieren?

Für die Risikoliste wird TabPFN verwendet, sofern es erfolgreich ausgeführt wurde.  
Andernfalls wird die Baseline genutzt.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TabPFN final trainieren
print("Trainiere TabPFN für die finale Risikoliste...")
classifier.fit(X_train, y_train)
finale_prognose = classifier.predict(X_test)
finales_risiko = classifier.predict_proba(X_test)[:, 1]

# 3. Risikoliste zusammenbauen
risikoliste = X_test.copy()
risikoliste["tatsaechlich_kuendigung"] = y_test.values
risikoliste["vorhergesagte_kuendigung"] = finale_prognose
risikoliste["kuendigungsrisiko_prozent"] = (finales_risiko * 100).round(1)

risikoliste = risikoliste.sort_values("kuendigungsrisiko_prozent", ascending=False)

# 4. Die wichtigsten deutschen Spalten für die Top-10-Ansicht
anzeigespalten = [
    "kuendigungsrisiko_prozent", 
    "Vertragsart",         
    "Vertragsdauer_Monate",
    "Monatliche_Kosten_Euro",
    "Internetdienst",      
    "Zahlungsmethode"      
]

print("\n🎯 Top 10 gefährdete Kunden (Action-Item für den Vertrieb):")
display(risikoliste[anzeigespalten].head(10))


# Die Top 10 für das Diagramm vorbereiten und für die Darstellung sortieren
top10 = risikoliste.head(10).sort_values("kuendigungsrisiko_prozent")

plt.figure(figsize=(9, 5))
plt.barh(
    [f"Kunde #{idx}" for idx in top10.index],
    top10["kuendigungsrisiko_prozent"],
)
plt.title("🎯 Zehn höchste Kündigungsrisiken (Top 10 Action-Item)")
plt.xlabel("Vorhergesagtes Kündigungsrisiko in Prozent")
plt.ylabel("Kunden-ID (Tabellen-Index)")
plt.xlim(0, 100)
plt.grid(axis="x", alpha=0.25)

# Die Prozentwerte direkt an die Balken schreiben
for index, wert in enumerate(top10["kuendigungsrisiko_prozent"]):
    plt.text(wert + 1, index, f"{wert:.1f} %", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()



## Wo macht das Modell Fehler?

Die Konfusionsmatrix zeigt:

- korrekt erkannte bleibende Kunden  
- korrekt erkannte Kündigungen  
- Fehlalarme  
- übersehene Kündigungen  

Gerade für Unternehmen ist wichtig, welche Fehlerart teurer ist.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    finale_prognose,
    display_labels=["bleibt", "kündigt"],
    values_format="d",
)

plt.title(f"Konfusionsmatrix – TabPFN")
plt.tight_layout()
plt.show()

# Schlussfolgerung für den Vortrag

**Was TabPFN vereinfacht**

- kein eigenes neuronales Netz entwickeln  
- wenig Modellkonfiguration  
- schnelle Anwendung auf kleine tabellarische Datensätze  
- vertraute Scikit-learn-Schnittstelle  

**Was trotzdem Aufgabe des Unternehmens bleibt**

- sinnvolle Geschäftsfrage auswählen  
- Datenqualität prüfen  
- ungeeignete Merkmale und Datenlecks vermeiden  
- Vorhersagen richtig bewerten  
- konkrete Maßnahmen aus zuverlässige KI ableiten  

> Eine hohe Kündigungswahrscheinlichkeit ist keine Gewissheit, sondern ein Hinweis darauf, wo sich ein genauerer Blick lohnen kann.

# Regression

## Datensatz laden und bereinigen

In [ ]:
from tabpfn import TabPFNRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 
from pathlib import Path
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing(as_frame=True)

housing_mapping = {
    "MedInc": "Mittleres_Einkommen",
    "HouseAge": "Hausalter_Jahre",
    "AveRooms": "Durchschnitt_Zimmer",
    "AveBedrms": "Durchschnitt_Schlafzimmer",
    "Population": "Bevoelkerung_Gebiet",
    "AveOccup": "Durchschnitt_Haushaltsgroesse",
    "Latitude": "Breitengrad",
    "Longitude": "Laengengrad",
    "MedHouseValue": "Immobilienpreis"
}

data2 = california_housing.data.rename(columns=housing_mapping)
data2['Immobilienpreis'] = california_housing.target

# Features und Zielvariable trennen
X_reg = data2.drop(columns=['Immobilienpreis'])
y_reg = data2['Immobilienpreis'] * 100000  # Umrechnung in echte Euro/Dollar-Werte

display(X_reg.head())


## TabPFN Reg trainieren

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# TabPFN Regressor initialisieren und trainieren
print("Trainiere TabPFN Regressor...")
regressor = TabPFNRegressor(ignore_pretraining_limits=True)
regressor.fit(X_train_r, y_train_r)

# Vorhersagen treffen
y_pred_reg = regressor.predict(X_test_r)

## Evaluation

In [ ]:
# Metriken berechnen
mae = mean_absolute_error(y_test_r, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_reg))
r2 = r2_score(y_test_r, y_pred_reg)

print(f"\n📊 Regressions-Auswertung für TabPFN:")
print(f"  • Mittlerer absoluter Fehler (MAE):  {mae:,.2f} €")
print(f"  • Wurzel des mittleren Fehlers (RMSE): {rmse:,.2f} €  (bestraft große Ausreißer stärker)")
print(f"  • Bestimmtheitsmaß (R² Score):       {r2:.3f}")

# Visueller Vergleich als Scatterplot
plt.figure(figsize=(7, 6))
plt.scatter(y_test_r, y_pred_reg, alpha=0.4, color="teal", edgecolors="none")

min_val = min(y_test_r.min(), y_pred_reg.min())
max_val = max(y_test_r.max(), y_pred_reg.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label="Perfekte Vorhersage")

plt.xlabel("Echter Immobilienpreis (€)")
plt.ylabel("Vorhergesagter Preis durch TabPFN (€)")
plt.title("TabPFN Regression: Echte vs. vorhergesagte Immobilienpreise")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

---
## 🛠️ Profi-Ausblick: Automatisierter Vergleich mit Kreuzvalidierung
In der realen Data-Science-Praxis verlässt man sich ungern auf *einen* zufälligen Train-Test-Split (vielleicht erwischt man zufällig einen "leichten" oder "schweren" Testdatensatz). 

Profis nutzen deshalb eine **Kreuzvalidierung (Cross-Validation)**: Das Modell wird mehrfach auf unterschiedlichen Teilen der Daten trainiert und die Leistung gemittelt. Hier siehst du, wie man mehrere Algorithmen (Random Forest, XGBoost, CatBoost und TabPFN) in einer eleganten Schleife gegeneinander antreten lässt:

In [ ]:
# 1. Benötigte Bibliotheken für XGBoost und CatBoost installieren (falls noch nicht geschehen)
%pip install xgboost catboost

# 2. Imports für die zusätzlichen Modelle und die Kreuzvalidierung
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ziel-Label sicherheitshalber für numerische Metriken encodieren
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Mehrere Modelle in einer Liste definieren
models = [
    ("TabPFN", classifier),  # Dein initialisiertes TabPFN-Modell
    (
        "RandomForest",
        make_pipeline(
            column_transformer,  
            RandomForestClassifier(random_state=42),
        ),
    ),
    (
        "XGBoost",
        make_pipeline(
            column_transformer,  
            XGBClassifier(random_state=42),
        ),
    ),
    (
        "CatBoost",
        make_pipeline(
            column_transformer,  
            CatBoostClassifier(random_state=42, verbose=0),
        ),
    ),
]

# Kreuzvalidierung einrichten (Stratified sorgt für ausgeglichene Klassen)
n_splits = 3
cv = StratifiedKFold(n_splits=n_splits, random_state=42, shuffle=True)
scoring = "roc_auc_ovr" if len(np.unique(y_encoded)) > 2 else "roc_auc"

print("⏳ Starte automatisierte Kreuzvalidierung über alle Modelle...")
scores = {
    name: cross_val_score(model, X, y_encoded, cv=cv, scoring=scoring, n_jobs=1, verbose=1)
    for name, model in models
}

# Ergebnisse als DataFrame aufbereiten und plotten
df_cv = pd.DataFrame(
    [(k, v.mean()) for (k, v) in scores.items()], columns=["Model", "ROC AUC"]
)

ax = df_cv.plot(x="Model", y="ROC AUC", kind="bar", figsize=(10, 5), color='mediumseagreen')
ax.set_ylim(df_cv["ROC AUC"].min() * 0.99, min(1.0, df_cv["ROC AUC"].max() * 1.01))
ax.set_title(f"Profi-Modellvergleich - {n_splits}-fold Cross-Validation (ROC AUC)")
ax.set_ylabel("ROC AUC Score")
plt.show()